# 01: Player Archetypes, Data Preparation

This notebook builds the player-season feature table used for archetypal analysis (AA).
It loads `players_full`, applies the sample filters, flags low-information rows, and builds
the 32 style-only features. Findings and decisions from the exploratory work are recorded
in the **Notes** section at the end.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# paths.py lives in the repo root, one level up from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import paths

players_full = paths.load(paths.PLAYERS_FULL)
print(f"players_full: {players_full.shape[0]:,} rows x {players_full.shape[1]:,} columns")

## 2. Column names and settings

In [ ]:
# Key columns
MIN_TOTAL_COL = "Min"     # season total minutes
FGA_COL = "FGA"           # season total field goal attempts
ID_COLS = ["Player ID", "Season"]

# Sample settings (see Notes for how these were chosen)
MIN_MINUTES = 250                    # default minimum minutes in a season
SEASON_MIN_MINUTES = {"2020-21": 200}  # shortened COVID season
LIMITED_GAMES_MIN_G = 8              # limited-games rule: minimum games...
LIMITED_GAMES_MIN_MPG = 25           # ...at this many minutes per game
MIN_PG_COL = "Min/G"
GP_COL = "G"
DROP_SEASONS = ["2018-19"]
LOW_FGA = 50
LOW_COV = 0.8

# Each player-season should appear exactly once
assert not players_full.duplicated(subset=ID_COLS).any(), "Duplicate player-seasons found"

## 3. Sample filters

In [ ]:
n_start = len(players_full)

# 1. Minutes threshold, with two exceptions
season_min = players_full["Season"].map(SEASON_MIN_MINUTES).fillna(MIN_MINUTES)
meets_minutes = players_full[MIN_TOTAL_COL] >= season_min
limited_games = (
  (players_full[GP_COL] >= LIMITED_GAMES_MIN_G)
  & (players_full[MIN_PG_COL] >= LIMITED_GAMES_MIN_MPG)
  & ~meets_minutes
)

pf = players_full[meets_minutes | limited_games].copy()
pf["keep_reason"] = "minutes"
pf.loc[limited_games[limited_games].index, "keep_reason"] = "limited_games"
pf.loc[(pf["Season"].isin(SEASON_MIN_MINUTES)) & (pf[MIN_TOTAL_COL] < MIN_MINUTES)
       & (pf["keep_reason"] == "minutes"), "keep_reason"] = "covid_threshold"
n_minutes = len(pf)

print("Rows kept by rule:")
print(pf["keep_reason"].value_counts())

# 2. Drop seasons with incomplete play-by-play tagging
pf = pf[~pf["Season"].isin(DROP_SEASONS)].copy()
n_seasons = len(pf)

# 3. Drop player-seasons with no shot zone data
basic5 = ["Rim Shot %", "Paint Shot %", "Midrange Shot %", "Corner 3 Shot %", "Above Break 3 Shot %"]
pf = pf[pf[basic5].sum(axis=1) >= 99].copy()
n_final = len(pf)

print(f"Start:                          {n_start:,}")
print(f"After {MIN_MINUTES}+ minutes:             {n_minutes:,}")
print(f"After dropping {DROP_SEASONS}:  {n_seasons:,}")
print(f"After dropping missing zones:   {n_final:,}\n")
print(pf["Season"].value_counts().sort_index())

## 4. Flags for low-information rows

In [ ]:
# Few shots: zone shares rest on a small sample
pf["low_fga"] = pf[FGA_COL] < LOW_FGA

# Incomplete play-by-play tagging: assisted and passing shares rest on fewer plays
zone_ast = ["Rim Ast", "Paint Ast", "Midrange Ast", "Corner 3 Ast", "Above Break 3 Ast"]
pf["make_tag_cov"] = (pf["FGM Assisted"] + pf["FGM Unassisted"]) / pf["FGM"]
pf["ast_tag_cov"] = pf[zone_ast].sum(axis=1) / pf["Ast"]
pf["low_coverage"] = (pf["make_tag_cov"] < LOW_COV) | (pf["ast_tag_cov"] < LOW_COV)

print(f"low_fga:      {pf['low_fga'].sum():,} player-seasons")
print(f"low_coverage: {pf['low_coverage'].sum():,} player-seasons\n")
print("low_coverage by season:")
print(pf.groupby("Season")["low_coverage"].agg(["sum", "mean"]).round(3))

## 5. Style features

In [ ]:
feat = pd.DataFrame(index=pf.index)

# Location (7): left and right sides combined
feat["Rim"] = pf["Rim Shot %"]
feat["Paint"] = pf["Paint Shot %"]
for zone in ["Baseline 2", "Elbow 2", "Corner 3", "Wing 3"]:
  feat[zone] = pf[f"Left {zone} FGA Freq %"] + pf[f"Right {zone} FGA Freq %"]
feat["Top of Key 3"] = pf["Top of Key 3 FGA Freq %"]

# Deep shooting (1)
feat["NBA 3"] = pf["NBA 3 FGA Freq %"]

# Shot type (5)
feat["Dunk"] = pf["Dunk FGA Freq %"]
feat["Layup"] = pf["Layup FGA Freq %"]
feat["Alley-Oop"] = pf["Alley-Oop FGA Freq %"]
feat["Putback"] = pf["FGA Freq % Putback"]
feat["Transition"] = pf["Transition Shot %"]

# Self-creation (2): share of made shots that were assisted (over tagged makes)
feat["2PM Assisted"] = pf["2PM Assisted %"]
feat["3PM Assisted"] = pf["3PM Assisted %"]

# Rim contact (1): shooting fouls drawn per rim shot attempt
feat["Rim Fouls Drawn"] = 100 * pf["Rim Shooting Foul Drawn"] / pf["Rim Shot Att"]

# Passing destination (5): stored as fractions, converted to 0-100
for zone in ["Rim", "Paint", "Midrange", "Corner 3", "Above Break 3"]:
  feat[f"Ast to {zone}"] = 100 * pf[f"% Ast {zone}"]

# Role (11)
for col in ["Usage %", "Ast %", "TO %", "OReb %", "DReb %", "Stl %", "Blk %",
            "FT Rate", "PF %", "Self OReb Share", "Blocked Per FGA"]:
  feat[col] = pf[col]

# Identifiers and flags kept alongside the features
ids = pf[ID_COLS + ["low_fga", "low_coverage", "keep_reason"]]

print(f"Feature table: {feat.shape[0]:,} rows x {feat.shape[1]} features")

## 6. Missing values and scale check

In [ ]:
missing = feat.isna().sum()
print("Missing values per feature:")
print(missing[missing > 0])

print("\nScale check (all should be roughly 0-100):")
with pd.option_context("display.max_rows", 50):
  print(feat.describe().loc[["min", "max"]].round(2).T)

## Notes: findings and decisions from early EDA

### Data structure
- `players_full` has 39,597 player-seasons (2018-19 to 2025-26) and 881 columns. `Player ID` plus `Season` is a unique key (no player has more than one row per season).
- `Min` and `FGA` are season totals (max 1,428 minutes and 712 FGA), not per-game averages.
- The data dictionary (`data_dictionary.csv`) groups `players_full` columns into 15 sections. Style features came mainly from Core, Shot Zones, Shot Types & Context, Assisted vs Unassisted, Rates & Percentages, and Rebounding Detail.

### Minutes threshold: 250 minutes, with two exceptions
- **Default:** 250+ minutes in a season.
- **COVID season (2020-21):** 200+ minutes, since canceled games reduced minutes without reducing roles. This keeps the season representative; noisy low-shot players are still caught by the `low_fga` flag.
- **Limited games:** 8+ games at 25+ minutes per game, to keep core rotation players who missed games (injury, suspension, eligibility). Their per-shot shares are meaningful, but they rest on fewer games, so they are labeled `keep_reason = "limited_games"` and are candidates for fit-then-score.
- Every kept row is labeled with `keep_reason` (`minutes`, `covid_threshold`, or `limited_games`).

### Low shot volume: flagged, not dropped
- At 250 minutes, the median player has 175 FGA; 702 player-seasons have fewer than 50 FGA (2.9%).
- These are kept and flagged (`low_fga`) because rarely shooting is itself a style (defensive specialists, low-usage bigs). Their zone shares are noisy, so plan to either shrink them toward the league average or exclude them from fitting and score them on the fitted archetypes.

### 2018-19 dropped
2018-19 failed nearly every play-by-play check:
- Only about 25% of made shots and assists are tagged (assisted/unassisted, assist zone) for the median player.
- Dunks are untagged (median dunk share 0.0% vs about 1.5% in other seasons).
- Shot-clock splits cover only about 25% of FGA.
- Steal locations (inside vs beyond the arc) are missing for 97% of players.
- The NCAA men's three-point line moved back starting in 2019-20, so 2018-19 three-point zones cover a different area.

### Play-by-play coverage after 2018-19
- Tagging improves gradually: median coverage is about 94% in 2019-20 and 100% from 2020-21 on, but the bottom 10% of players have under 46% (2019-20) and 80% (2020-21) of their makes tagged. From 2021-22 on, nearly everyone is above 92%.
- Assisted shares (`2PM Assisted %` etc.) and passing shares are computed over **tagged** plays only (verified), so incomplete tagging adds noise but not bias.
- Players under 80% coverage are flagged (`low_coverage`) for the same treatment options as `low_fga`.

### Verified column meanings
- `Rim Assisted %` (and the other zone `Assisted %` columns): share of the player's own makes in that zone that were assisted (matches assisted makes / makes to within 0.05).
- `% Ast Rim` (and other `% Ast` zone columns): share of the player's assists that led to shots in that zone. Stored as a fraction (0-1), unlike most of the package (0-100); multiplied by 100 in the feature table.
- Detailed zones are exact subdivisions of the basic five: baseline 2 + elbow 2 = midrange, and wings + top of key = above-the-break 3 (to within 0.1).
- Basic five zone shares sum to about 100; 214 player-seasons (under 1%) have no zone data and are dropped.

### Feature set (32 features)
| Group | Features |
|---|---|
| Location (7) | Rim, Paint, Baseline 2, Elbow 2, Corner 3, Wing 3, Top of Key 3 |
| Deep shooting (1) | NBA 3 |
| Shot type (5) | Dunk, Layup, Alley-Oop, Putback, Transition |
| Self-creation (2) | 2PM Assisted, 3PM Assisted |
| Rim contact (1) | Rim Fouls Drawn (per rim shot attempt) |
| Passing (5) | Ast to Rim, Paint, Midrange, Corner 3, Above Break 3 |
| Role (11) | Usage %, Ast %, TO %, OReb %, DReb %, Stl %, Blk %, FT Rate, PF %, Self OReb Share, Blocked Per FGA |

**Rule for inclusion:** style is what a player does and how often; quality is how often it works. Shares and activity rates are included; success rates (FG%, eFG%, TS%, FT%, points per shot) are excluded.

**Judgment calls to state in Methods:**
- `TO %` is included because turnovers mainly reflect exposure to on-ball actions, although it also partly reflects decision-making quality.
- `Self OReb Share` overlaps with `OReb %` and putbacks; `Blocked Per FGA` partly reflects shot selection.
- Plan: fit AA with and without these three (and with and without height) and compare the archetypes.

### Considered and excluded
- **Efficiency stats** (FG%, eFG%, TS%, FT%, Pts/Shot): quality, not style.
- **Distance bands and left/center/right splits:** duplicate the zone shares.
- **Heave 3 and Behind Hoop:** desperation or very rare shots.
- **Layup Dunk combined, 3PA Rate %:** sums of included features.
- **Shot-clock splits (S01/S12/S23):** incomplete coverage in early seasons; could return now that 2018-19 is dropped (2019-20 coverage is about 93%).
- **Steal location (Stl2/Stl3):** coverage incomplete until 2024-25.
- **Charges drawn:** could not be derived; `Blk Rcv` does not match recovered blocks or blocks received, so the formula (stops minus recovered blocks minus steals) gave 3,802 negative values.
- **Individual Stops / Stop %:** sum of recovered blocks, steals, and charges; mostly repeats Stl % and Blk %.
- **Jailbreak Steal %:** measures converting steals into points (success), and depends on teammates.
- **Assisted Makes %:** overlaps the 2PM/3PM versions.
- **Min Share %:** reflects playing time and coach trust, not style.